# V5 v2 Residual Listwise Ranker

Train a second-stage ranker on cached candidates from the serving V4 configuration (`popularity_blend_weight=0.02`). Candidate retrieval runs once, then all epochs reuse the cache. The ranker starts as an identity ordering of V4 and is selected by validation multi-positive NDCG@10, not training loss.

In [ ]:
from google.colab import drive, files, userdata
drive.mount('/content/drive')

from pathlib import Path
import gc, hashlib, importlib.util, json, shutil, subprocess, sys, zipfile

DRIVE_ROOT = Path('/content/drive/MyDrive/recsys')
ADAPTIVE_ROOT = DRIVE_ROOT / 'data/gold/adaptive/v1'
BASE_GOLD_ROOT = DRIVE_ROOT / 'data/gold/two_tower/v1_colab'
WORK_ROOT = Path('/content/recsys_adaptive_eval')
SOURCE_ROOT = WORK_ROOT / 'runtime_v2'
V4_DIR = WORK_ROOT / 'artifacts/v4_sequence_retrieval'
V5_V2_DIR = WORK_ROOT / 'artifacts/v5_ranker_v2'
EVAL_DIR = WORK_ROOT / 'reports/v5_ranker_v2'
for path in [SOURCE_ROOT, V4_DIR, V5_V2_DIR, EVAL_DIR]:
    path.mkdir(parents=True, exist_ok=True)
print('workspace:', WORK_ROOT)

In [ ]:
required = {'pyarrow': 'pyarrow', 'pandas': 'pandas', 'torch': 'torch', 'huggingface_hub': 'huggingface_hub>=0.32'}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
import torch
assert torch.cuda.is_available(), 'Select a Colab GPU runtime. L4 or A100 with High-RAM is recommended.'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
required_paths = [
    ADAPTIVE_ROOT / 'sequence_examples/train',
    ADAPTIVE_ROOT / 'sequence_examples/validation',
    BASE_GOLD_ROOT / 'transforms/numeric_stats.json',
    BASE_GOLD_ROOT / 'vocabularies/user_active_degree',
    BASE_GOLD_ROOT / 'vocabularies/video_type',
    BASE_GOLD_ROOT / 'vocabularies/upload_type',
]
missing = [str(path) for path in required_paths if not path.exists()]
assert not missing, 'Missing inputs:\n' + '\n'.join(missing)
assert (ADAPTIVE_ROOT / 'sequence_examples/train/_SUCCESS').exists()
assert (ADAPTIVE_ROOT / 'sequence_examples/validation/_SUCCESS').exists()
print('Drive inputs ready')

## Download private runtime and V4 checkpoint

No browser upload is required. Add a Hugging Face token with access to the private `khoa1505/recsys-artifacts` repository to Colab Secrets using the name `HF_TOKEN`. This cell downloads the small runtime and the verified FP16 V4 checkpoint directly from Hugging Face.

In [ ]:
from huggingface_hub import hf_hub_download

HF_REPO_ID = 'khoa1505/recsys-artifacts'
V4_FILENAME = 'v4/v4_inference_fp16.pt'
V4_EXPECTED_SHA256 = '4ad49b3e6f9f8b5d45a6c92f31d8d9b27f558e1746613efb44a23db3cd456a26'
RUNTIME_FILENAME = 'runtime/adaptive_eval_runtime.zip'
RUNTIME_EXPECTED_SHA256 = '1262393d30e21082849da6ae9ff74f1da4f4eac1d112860dac6a071909182327'
try:
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception as exc:
    raise RuntimeError('Add a Colab Secret named HF_TOKEN and grant this notebook access to it.') from exc
assert HF_TOKEN, 'HF_TOKEN is empty. Add a read token in Colab Secrets.'
runtime_zip = Path(hf_hub_download(repo_id=HF_REPO_ID, filename=RUNTIME_FILENAME, token=HF_TOKEN))
v4_cached = Path(hf_hub_download(repo_id=HF_REPO_ID, filename=V4_FILENAME, token=HF_TOKEN))
v4_manifest_path = Path(hf_hub_download(repo_id=HF_REPO_ID, filename='v4/manifest.json', token=HF_TOKEN))
print('downloaded runtime:', runtime_zip)
print('downloaded V4:', v4_cached, f'{v4_cached.stat().st_size / 1024**3:.2f} GiB')

if SOURCE_ROOT.exists():
    shutil.rmtree(SOURCE_ROOT)
SOURCE_ROOT.mkdir(parents=True)
with zipfile.ZipFile(runtime_zip) as archive:
    archive.extractall(SOURCE_ROOT)
if str(SOURCE_ROOT) not in sys.path:
    sys.path.insert(0, str(SOURCE_ROOT))
assert (SOURCE_ROOT / 'scripts/train_adaptive_v5_ranker_v2.py').exists()
v4_target = V4_DIR / 'best_model.pt'
if v4_target.exists() or v4_target.is_symlink():
    v4_target.unlink()
v4_target.symlink_to(v4_cached.resolve())
print('V5 v2 runtime and V4 checkpoint ready')

## Artifact integrity preflight

Verify both downloaded files before loading any model weights. The expected hashes are pinned in this notebook and also stored in the private repository manifests.

In [ ]:
def sha256(path: Path, block_size: int = 8 * 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        while block := stream.read(block_size):
            digest.update(block)
    return digest.hexdigest()

assert sha256(runtime_zip) == RUNTIME_EXPECTED_SHA256, 'Runtime ZIP hash mismatch'
assert sha256(v4_cached) == V4_EXPECTED_SHA256, 'V4 checkpoint hash mismatch'
v4_manifest = json.loads(v4_manifest_path.read_text())
v4_entry = next(item for item in v4_manifest['files'] if item['filename'] == 'v4_inference_fp16.pt')
assert v4_entry['bytes'] == v4_cached.stat().st_size
assert v4_entry['sha256'] == V4_EXPECTED_SHA256
checkpoint = torch.load(v4_target, map_location='cpu', weights_only=False)
required_keys = {'config', 'model_state_dict', 'vocab_sizes'}
assert required_keys.issubset(checkpoint), f'Missing V4 checkpoint keys: {required_keys - set(checkpoint)}'
assert len(checkpoint['model_state_dict']) == v4_manifest['state_dict_tensor_count'] == 54
assert all(torch.isfinite(tensor).all().item() for tensor in checkpoint['model_state_dict'].values() if tensor.is_floating_point())
del checkpoint
gc.collect()
print('artifact integrity: PASS')
print('V4 SHA256:', V4_EXPECTED_SHA256)

## Train V5 v2

The expensive V4 item embeddings and candidates are generated once. Training then reuses those arrays for every epoch. The initial epoch-0 checkpoint preserves V4 ordering exactly.

In [ ]:
train_script = SOURCE_ROOT / 'scripts/train_adaptive_v5_ranker_v2.py'
command = [
    sys.executable, str(train_script),
    '--adaptive-root', str(ADAPTIVE_ROOT),
    '--base-gold-root', str(BASE_GOLD_ROOT),
    '--v4-dir', str(V4_DIR),
    '--output-dir', str(V5_V2_DIR),
    '--max-train-queries', '80000',
    '--max-validation-users', '0',
    '--train-candidates', '128',
    '--validation-candidates', '1000',
    '--popularity-blend-weight', '0.02',
    '--epochs', '12',
    '--batch-query-size', '64',
    '--retrieval-query-batch-size', '32',
    '--learning-rate', '0.0003',
    '--early-stopping-patience', '3',
    '--device', 'cuda',
]
def run_with_live_log(command, log_path: Path):
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    with log_path.open('w') as log_file:
        for line in process.stdout:
            print(line, end='')
            log_file.write(line)
            log_file.flush()
    return_code = process.wait()
    if return_code != 0:
        tail = '\n'.join(log_path.read_text(errors='replace').splitlines()[-120:])
        raise RuntimeError(f'Command failed with exit code {return_code}. Last log lines:\n{tail}')

run_with_live_log(command, Path('/content/v5_v2_training.log'))
metrics = json.loads((V5_V2_DIR / 'metrics.json').read_text())
print('accepted_over_v4:', metrics['accepted_over_v4'])
print('best_epoch:', metrics['best_epoch'])
print('best_validation_ndcg@10:', metrics['best_validation_ndcg@10'])

## Controlled full validation evaluation

This compares V5 v2 against the same V4 blend-0.02 Top-1000 candidates.

In [ ]:
evaluate_script = SOURCE_ROOT / 'scripts/evaluate_adaptive_v4_v5.py'
command = [
    sys.executable, str(evaluate_script),
    '--adaptive-root', str(ADAPTIVE_ROOT),
    '--base-gold-root', str(BASE_GOLD_ROOT),
    '--v4-dir', str(V4_DIR),
    '--v5-dir', str(V5_V2_DIR),
    '--output-dir', str(EVAL_DIR),
    '--split', 'validation',
    '--max-queries', '0',
    '--max-candidate-items', '0',
    '--retrieval-candidates', '1000',
    '--v5-candidate-blend-weight', '0.02',
    '--popularity-blend-grid', '0.0,0.02',
    '--query-batch-size', '32',
    '--device', 'cuda',
]
run_with_live_log(command, Path('/content/v5_v2_evaluation.log'))
evaluation = json.loads((EVAL_DIR / 'adaptive_v4_v5_evaluation.json').read_text())
evaluation['ranker_delta_vs_v4_multi_positive']

## Persist and download V5 v2 artifacts

The archive is small: checkpoint, config, metrics, history, logs, and controlled evaluation only. It is copied to Drive before the optional browser download, so a browser download failure does not lose the completed run. Candidate caches are intentionally excluded.

In [ ]:
artifact_zip = Path('/content/v5_ranker_v2_artifacts.zip')
if artifact_zip.exists():
    artifact_zip.unlink()
with zipfile.ZipFile(artifact_zip, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for root in [V5_V2_DIR, EVAL_DIR]:
        for path in root.rglob('*'):
            if path.is_file():
                archive.write(path, path.relative_to(WORK_ROOT))
    for log_path in [Path('/content/v5_v2_training.log'), Path('/content/v5_v2_evaluation.log')]:
        if log_path.exists():
            archive.write(log_path, Path('logs') / log_path.name)
persistent_dir = DRIVE_ROOT / 'artifacts/adaptive/v5_ranker_v2'
persistent_dir.mkdir(parents=True, exist_ok=True)
persistent_zip = persistent_dir / artifact_zip.name
shutil.copy2(artifact_zip, persistent_zip)
print('artifact archive:', artifact_zip, artifact_zip.stat().st_size, 'bytes')
print('persistent Drive copy:', persistent_zip)
files.download(str(artifact_zip))